In [ ]:
# Cell 1 — Install dependencies & download arXiv dataset
# Enable GPU: Notebook Settings → Accelerator → GPU T4 x2

# Install fine-tuning stack
!pip install -q unsloth trl transformers datasets accelerate bitsandbytes

# Install Kaggle API (already available on Kaggle, but ensure it's configured)
import os
import json

# Set up Kaggle credentials from Kaggle Secrets
# Add KAGGLE_USERNAME and KAGGLE_KEY as Kaggle Secrets before running
from kaggle_secrets import UserSecretsClient  # type: ignore
secrets = UserSecretsClient()
os.environ['KAGGLE_USERNAME'] = secrets.get_secret('KAGGLE_USERNAME')
os.environ['KAGGLE_KEY'] = secrets.get_secret('KAGGLE_KEY')

# Download arXiv metadata snapshot (~4 GB compressed)
!kaggle datasets download -d Cornell-University/arxiv -p /kaggle/working/ --unzip

print('Setup complete. Dataset downloaded to /kaggle/working/')
!ls -lh /kaggle/working/*.json

In [ ]:
# Cell 2 — Prepare dataset (filter CS/AI papers, build training pairs)

import json
import random
import re
import os

SNAPSHOT_PATH = '/kaggle/working/arxiv-metadata-oai-snapshot.json'
TRAIN_PATH = '/kaggle/working/train.jsonl'
VAL_PATH = '/kaggle/working/val.jsonl'
TARGET_CATEGORIES = ('cs.', 'stat.ML')
MIN_ABSTRACT_CHARS = 100
MAX_SAMPLES = 10_000
TRAIN_SIZE = 9_000
RANDOM_SEED = 42

def _clean(text):
    text = text.replace('\n', ' ')
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

def _is_target(entry):
    cats = entry.get('categories', '')
    return any(t in cats for t in TARGET_CATEGORIES)

def _build_sample(entry):
    title = _clean(entry.get('title', ''))
    abstract = _clean(entry.get('abstract', ''))
    prompt = 'Analyze this research paper abstract and explain it clearly:\n\n' + abstract
    response = f'{title} - {abstract}'
    return {'prompt': prompt, 'response': response}

print('Scanning arXiv snapshot …')
candidates = []
total_lines = 0

with open(SNAPSHOT_PATH, 'r', encoding='utf-8') as fh:
    for line in fh:
        total_lines += 1
        if total_lines % 500_000 == 0:
            print(f'  {total_lines:,} lines scanned, {len(candidates):,} candidates')
        line = line.strip()
        if not line:
            continue
        try:
            entry = json.loads(line)
        except json.JSONDecodeError:
            continue
        if not _is_target(entry):
            continue
        abstract = _clean(entry.get('abstract', ''))
        if len(abstract) < MIN_ABSTRACT_CHARS:
            continue
        candidates.append(entry)

print(f'Found {len(candidates):,} candidates from {total_lines:,} total lines.')

random.seed(RANDOM_SEED)
random.shuffle(candidates)
selected = candidates[:MAX_SAMPLES]
samples = [_build_sample(e) for e in selected]

with open(TRAIN_PATH, 'w', encoding='utf-8') as fh:
    for s in samples[:TRAIN_SIZE]:
        fh.write(json.dumps(s, ensure_ascii=False) + '\n')

with open(VAL_PATH, 'w', encoding='utf-8') as fh:
    for s in samples[TRAIN_SIZE:]:
        fh.write(json.dumps(s, ensure_ascii=False) + '\n')

print(f'train.jsonl: {TRAIN_SIZE:,} samples')
print(f'val.jsonl:   {MAX_SAMPLES - TRAIN_SIZE:,} samples')

In [ ]:
# Cell 3 — Fine-tune llama3.1:8b with Unsloth + LoRA

import os
from unsloth import FastLanguageModel
from datasets import load_dataset
from trl import SFTTrainer
from transformers import TrainingArguments, TrainerCallback

OUTPUT_DIR = '/kaggle/working/finetuned_model'
LOG_FILE = '/kaggle/working/training_log.txt'
TRAIN_PATH = '/kaggle/working/train.jsonl'
VAL_PATH = '/kaggle/working/val.jsonl'

ALPACA_TEMPLATE = (
    'Below is an instruction that describes a task. '
    'Write a response that appropriately completes the request.\n\n'
    '### Instruction:\n{prompt}\n\n### Response:\n{response}'
)

# Load model
print('Loading model …')
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name='unsloth/Meta-Llama-3.1-8B-bnb-4bit',
    max_seq_length=2048,
    load_in_4bit=True,
    dtype=None,
)

# Apply LoRA
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    lora_alpha=32,
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'],
    lora_dropout=0.05,
    bias='none',
    use_gradient_checkpointing='unsloth',
    random_state=42,
)

# Load dataset
dataset = load_dataset('json', data_files={'train': TRAIN_PATH, 'validation': VAL_PATH})
eos_token = tokenizer.eos_token or '<|end_of_text|>'

def format_sample(s):
    text = ALPACA_TEMPLATE.format(prompt=s['prompt'], response=s['response']) + eos_token
    return {'text': text}

dataset = dataset.map(format_sample, remove_columns=['prompt', 'response'])

# Loss logger
class LossCallback(TrainerCallback):
    def __init__(self):
        with open(LOG_FILE, 'w') as f:
            f.write('step,loss\n')
    def on_log(self, args, state, control, logs=None, **kwargs):
        if logs and 'loss' in logs:
            with open(LOG_FILE, 'a') as f:
                f.write(f"{state.global_step},{logs['loss']:.6f}\n")

# Train
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=3,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    warmup_steps=50,
    logging_steps=50,
    save_steps=500,
    fp16=True,
    optim='adamw_8bit',
    lr_scheduler_type='cosine',
    report_to='none',
    evaluation_strategy='epoch',
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset['train'],
    eval_dataset=dataset['validation'],
    dataset_text_field='text',
    max_seq_length=2048,
    args=training_args,
    callbacks=[LossCallback()],
)

trainer.train()
trainer.model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f'Model saved to {OUTPUT_DIR}')

In [ ]:
# Cell 4 — Copy model to Kaggle output panel for download

import shutil
import os

SRC = '/kaggle/working/finetuned_model'
DST = '/kaggle/output/finetuned_model'

os.makedirs('/kaggle/output', exist_ok=True)

if os.path.exists(DST):
    shutil.rmtree(DST)

shutil.copytree(SRC, DST)
print(f'Copied model to output panel: {DST}')

# Also copy training log
log_src = '/kaggle/working/training_log.txt'
if os.path.exists(log_src):
    shutil.copy(log_src, '/kaggle/output/training_log.txt')
    print('Copied training_log.txt')

# List output
print('\nOutput directory contents:')
for name in sorted(os.listdir(DST)):
    size = os.path.getsize(os.path.join(DST, name))
    print(f'  {name:<40} {size / 1e6:.1f} MB')

print('\nDownload from the Output tab on the right side of the Kaggle notebook UI.')

In [ ]:
# Cell 5 — Quick inference test on the fine-tuned model

from transformers import pipeline
import torch

MODEL_DIR = '/kaggle/working/finetuned_model'

print('Loading fine-tuned model for inference test …')
pipe = pipeline(
    'text-generation',
    model=MODEL_DIR,
    torch_dtype=torch.float16,
    device_map='auto',
)

TEST_ABSTRACT = (
    'We propose a novel transformer architecture that achieves state-of-the-art '
    'performance on natural language inference benchmarks by incorporating '
    'cross-attention between task-specific and general-purpose encoders. '
    'Our method reduces parameter count by 40% while maintaining accuracy '
    'on GLUE and SuperGLUE tasks.'
)

prompt = (
    'Below is an instruction that describes a task. '
    'Write a response that appropriately completes the request.\n\n'
    '### Instruction:\n'
    'Analyze this research paper abstract and explain it clearly:\n\n'
    + TEST_ABSTRACT +
    '\n\n### Response:\n'
)

output = pipe(prompt, max_new_tokens=256, temperature=0.3, do_sample=True)
response = output[0]['generated_text'][len(prompt):].strip()

print('=== Model Response ===')
print(response)
print('=====================')